## Step 1: Mount Google Drive (if using Drive for files)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python-headless
!pip install tqdm
!pip install einops  # Required for NAFNet

## Step 3: Create Directory Structure

In [ ]:
!mkdir -p /content/nafnet_test/core
!mkdir -p /content/nafnet_test/weights
!mkdir -p /content/nafnet_test/input
!mkdir -p /content/nafnet_test/output

print("✅ Directory structure created")

## Step 4: Upload Required Files

Upload these files to the corresponding folders:

1. **nafnet_arch.py** → `/content/nafnet_test/core/`
2. **test_nafnet_video.py** → `/content/nafnet_test/`
3. **nafnet_wagon_finetuned.pth** → `/content/nafnet_test/weights/`
4. **your_video.mp4** → `/content/nafnet_test/input/`

OR use Google Drive paths (faster for large files)

In [ ]:
# Option A: Upload from local computer
from google.colab import files

print("Upload nafnet_arch.py:")
uploaded = files.upload()
!mv nafnet_arch.py /content/nafnet_test/core/

print("\nUpload test_nafnet_video.py:")
uploaded = files.upload()
!mv test_nafnet_video.py /content/nafnet_test/

print("\nUpload model weights (.pth file):")
uploaded = files.upload()
!mv *.pth /content/nafnet_test/weights/nafnet_wagon_finetuned.pth

print("\nUpload input video:")
uploaded = files.upload()
!mv *.mp4 /content/nafnet_test/input/input_video.mp4

In [ ]:
# Option B: Use Google Drive (recommended for large files)
# After mounting Drive, copy files from your Drive

# Adjust these paths to match your Google Drive structure
!cp "/content/drive/MyDrive/Motion blur mitigation/full model/src/core/nafnet_arch.py" /content/nafnet_test/core/
!cp "/content/drive/MyDrive/Motion blur mitigation/full model/src/scripts/test_nafnet_video.py" /content/nafnet_test/
!cp "/content/drive/MyDrive/Motion blur mitigation/full model/finetuned_nafnet/nafnet_wagon_finetuned.pth" /content/nafnet_test/weights/
!cp "/content/drive/MyDrive/your_video.mp4" /content/nafnet_test/input/input_video.mp4

print("✅ Files copied from Google Drive")

## Step 5: Verify Files

In [ ]:
!ls -lh /content/nafnet_test/core/
!ls -lh /content/nafnet_test/weights/
!ls -lh /content/nafnet_test/input/
!ls -lh /content/nafnet_test/

## Step 6: Check GPU Availability

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > GPU")

## Step 7: Run Deblurring

Choose your enhancement level:

In [ ]:
# Change to project directory
%cd /content/nafnet_test

In [ ]:
# OPTION 1: Basic deblurring (fastest)
!python test_nafnet_video.py \
    --input input/input_video.mp4 \
    --output output/deblurred_basic.mp4 \
    --model weights/nafnet_wagon_finetuned.pth

In [ ]:
# OPTION 2: With temporal stacking (better quality)
!python test_nafnet_video.py \
    --input input/input_video.mp4 \
    --output output/deblurred_temporal.mp4 \
    --model weights/nafnet_wagon_finetuned.pth \
    --temporal \
    --temporal-frames 5

In [ ]:
# OPTION 3: Full enhancement (best for OCR)
!python test_nafnet_video.py \
    --input input/input_video.mp4 \
    --output output/deblurred_enhanced.mp4 \
    --model weights/nafnet_wagon_finetuned.pth \
    --enhance-all \
    --sharpen 0.7

In [ ]:
# OPTION 4: Side-by-side comparison (test on 100 frames)
!python test_nafnet_video.py \
    --input input/input_video.mp4 \
    --output output/comparison.mp4 \
    --model weights/nafnet_wagon_finetuned.pth \
    --enhance-all \
    --save-comparison \
    --max-frames 100

In [ ]:
# OPTION 5: High quality with aligned temporal stacking (slowest, best quality)
!python test_nafnet_video.py \
    --input input/input_video.mp4 \
    --output output/deblurred_best.mp4 \
    --model weights/nafnet_wagon_finetuned.pth \
    --temporal \
    --temporal-method aligned \
    --temporal-frames 7 \
    --clahe \
    --denoise \
    --sharpen 0.8 \
    --tta

## Step 8: Preview Output (Optional)

In [ ]:
# Display first frame of output video
import cv2
from IPython.display import Image, display
import matplotlib.pyplot as plt

cap = cv2.VideoCapture('/content/nafnet_test/output/deblurred_enhanced.mp4')
ret, frame = cap.read()
cap.release()

if ret:
    plt.figure(figsize=(15, 10))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('First Frame - Deblurred Output')
    plt.show()
else:
    print("Could not read video")

## Step 9: Download Output Video

In [ ]:
# Option A: Download to your computer
from google.colab import files
files.download('/content/nafnet_test/output/deblurred_enhanced.mp4')

In [ ]:
# Option B: Save to Google Drive
!cp /content/nafnet_test/output/*.mp4 "/content/drive/MyDrive/deblurred_outputs/"
print("✅ Videos saved to Google Drive")

## Performance Tips

- **Free Colab**: T4 GPU (16GB VRAM) - Can handle Full HD videos
- **Colab Pro**: Better GPUs (A100, V100) - Faster processing
- **To speed up**: Use `--max-frames 100` to test on sample first
- **For long videos**: Process in chunks using `--skip-frames` and `--max-frames`